# Renombrador Masivo de Archivos

Proyecto en **Python puro** (`pathlib`).

Permite renombrar archivos con:
- Patrones (`{n}` para número, `{fecha}` para fecha)
- Filtro por extensión
- Prefijo y sufijo
- Modo simulación (seguro) y aplicación real

Ejecuta las celdas en orden.

In [ ]:
from datetime import datetime
from pathlib import Path

def renombrar_archivos(
    carpeta: str,
    patron: str = "archivo_{n}",
    extension: str = None,
    inicio: int = 1,
    incluir_fecha: bool = False,
    prefijo: str = "",
    sufijo: str = "",
    simulacion: bool = True,
) -> list:
    """Renombra archivos. Devuelve lista de (original, nuevo)."""
    carpeta_path = Path(carpeta)
    if not carpeta_path.exists() or not carpeta_path.is_dir():
        raise ValueError(f"Carpeta no válida: {carpeta}")

    archivos = [
        f for f in carpeta_path.iterdir()
        if f.is_file() and (not extension or f.suffix.lower() == extension.lower())
    ]
    archivos.sort()

    cambios = []
    fecha = datetime.now().strftime("%Y%m%d") if incluir_fecha else ""

    for i, archivo in enumerate(archivos, start=inicio):
        nuevo_nombre = patron.format(n=i, fecha=fecha)
        if prefijo:
            nuevo_nombre = prefijo + nuevo_nombre
        if sufijo:
            nuevo_nombre = nuevo_nombre + sufijo

        if not extension:
            nuevo_nombre += archivo.suffix

        nuevo_path = archivo.parent / nuevo_nombre
        cambios.append((archivo, nuevo_path))

    if not simulacion:
        for original, nuevo in cambios:
            if original != nuevo:
                original.rename(nuevo)

    return cambios

def mostrar_cambios(cambios: list, simulacion: bool = True):
    if not cambios:
        print("No hay archivos para renombrar.")
        return

    modo = "SIMULACIÓN" if simulacion else "EJECUTADO"
    print(f"\n[{modo}] {len(cambios)} archivo(s):\n")
    for original, nuevo in cambios:
        print(f"  {original.name}  →  {nuevo.name}")
    print()

print("Funciones cargadas: renombrar_archivos() y mostrar_cambios()")

## Ejemplo (solo simulación)
Cambia la carpeta y el patrón según necesites.

In [ ]:
# Simulación segura (no cambia nada)
cambios = renombrar_archivos(
    carpeta=".",
    patron="foto_{n}",
    extension=".jpg",
    inicio=1,
    incluir_fecha=True,
    simulacion=True
)
mostrar_cambios(cambios, simulacion=True)

# Para aplicar de verdad, cambia simulacion=False
# (ten cuidado, renombra de verdad)

## Modo interactivo
Primero muestra simulación y luego pregunta si quieres aplicar.

In [ ]:
print("=== Renombrador Masivo ===\n")
carpeta = input("Carpeta (Enter = actual): ").strip() or "."
patron = input("Patrón (usa {n} y {fecha}, defecto 'archivo_{n}'): ").strip() or "archivo_{n}"
extension = input("Filtrar por extensión (ej: .jpg, opcional): ").strip() or None
inicio = int(input("Número inicial (defecto 1): ") or 1)
incluir_fecha = input("Incluir fecha? (s/n): ").lower() == "s"
prefijo = input("Prefijo (opcional): ").strip()
sufijo = input("Sufijo (opcional): ").strip()

cambios = renombrar_archivos(carpeta, patron, extension, inicio, incluir_fecha, prefijo, sufijo, simulacion=True)
mostrar_cambios(cambios, simulacion=True)

if cambios:
    confirmar = input("¿Aplicar cambios? (s/n): ").lower()
    if confirmar == "s":
        renombrar_archivos(carpeta, patron, extension, inicio, incluir_fecha, prefijo, sufijo, simulacion=False)
        print("Cambios aplicados.")
    else:
        print("Cancelado.")